# Gray SOTA on identical draws (CPU): Xformer and GRL-B on Set12 / BSD68, seed 0

These are the released σ-specific gray denoisers, run on exactly the noisy images of `gray-lift-kill-test` (seed 0),
for paired tests against the chroma-noise lift (`results/gray_lift/scores.csv`).
* Protocol: Restormer / KAIR gray. The float32 image is divided by 255, `np.random.seed(0)` is set per image, then
  `img += np.random.normal(0, σ/255)`. No clipping; outputs are clamped and rounded to uint8.
* Xformer (ICLR 2024), gladzhang/Xformer @ ecc1e910. Arch flags and test padding (reflect to a multiple of 8) are
  copied from `evaluate/test_gaussian_gray_denoising.py`; weights `xformer_gray_dn_sigma{15,25,50}.pth` (key
  `params`) from the authors' Google Drive folder. Published Set12 33.46 / 31.16 / 28.10, BSD68 31.98 / 29.55 / 26.65.
* GRL-B (CVPR 2023), ofsoundof/GRL-Image-Restoration @ 3123fcad. Kwargs are `config/model/grl/grl_base.yaml` with
  the overrides of `config/experiment/dn/grl/grl_p256.yaml` and of the GRL-Base denoising command in
  `scripts/grl/grl_test.md` (`window_size=32 anchor_window_down_factor=2 tile=256 tile_overlap=32`); weights
  `v1.0.0/dn_grl_base_c1s{15,25,50}.ckpt`. Published Set12 33.47 / 31.12 / 28.03, BSD68 32.00 / 29.54 / 26.60.
* CPU budget: Xformer runs first (cheap), then GRL-B (≈40 transformer blocks at full resolution, tiled) with σ in
  the order 50, 25, 15 until an 11 h deadline. `outputs/scores.csv` is rewritten after every (model, dataset, σ).
Kaggle: CPU, internet on.

In [ ]:
import importlib.util, math, os, pickle, subprocess, sys, time, traceback, types, urllib.request
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

SIGMAS, SEED = [50, 25, 15], 0  # σ order: most informative first, in case the deadline cuts GRL-B short
RUN = ["Xformer"]                   # models run by this kernel; the two are split over two CPU kernels for time
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT, f"{CACHE}/xformer", f"{CACHE}/grl/models/common", f"{CACHE}/grl/models/networks"):
    os.makedirs(d, exist_ok=True)
torch.set_num_threads(os.cpu_count())
T_START = time.time()
DEADLINE_S = 11 * 3600  # stop starting new images after this (Kaggle CPU sessions die at 12 h)


def download(url, path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        for attempt in range(5):
            try:
                urllib.request.urlretrieve(url, path)
                break
            except Exception as e:
                print(f"  retry {attempt + 1} for {url}: {e}", flush=True)
                time.sleep(5 * (attempt + 1))
    return path


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


for pkg in ("gdown", "omegaconf", "einops", "timm"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
if importlib.util.find_spec("fairscale") is None:  # GRL imports checkpoint_wrapper; unused with fairscale_checkpoint=False
    sys.modules["fairscale"] = types.ModuleType("fairscale")
    sys.modules["fairscale.nn"] = types.ModuleType("fairscale.nn")
    sys.modules["fairscale.nn"].checkpoint_wrapper = lambda module, **kwargs: module
    sys.modules["fairscale"].nn = sys.modules["fairscale.nn"]

XFORMER = "https://raw.githubusercontent.com/gladzhang/Xformer/ecc1e910b923f35c8d46f5ffbc8ed2bc603f6567/{}"
GRL_SRC = "https://raw.githubusercontent.com/ofsoundof/GRL-Image-Restoration/3123fcadfee2107d813c5e8fff58227f14c989bc/{}"
GRL_REL = "https://github.com/ofsoundof/GRL-Image-Restoration/releases/download/v1.0.0/{}"
RAW = "https://raw.githubusercontent.com/cszn/DnCNN/master/testsets/{}/{}"
IMAGES = {"Set12": [f"{i:02d}.png" for i in range(1, 13)], "BSD68": [f"test{i:03d}.png" for i in range(1, 69)]}
jobs = {f"{CACHE}/xformer/x_former_arch.py": XFORMER.format("basicsr/models/archs/x_former_arch.py")}
jobs.update({f"{CACHE}/grl/models/common/{f}.py": GRL_SRC.format(f"models/common/{f}.py")
             for f in ("__init__", "resblock", "upsample", "ops", "swin_v1_block", "mixed_attn_block",
                       "mixed_attn_block_efficient")})
jobs[f"{CACHE}/grl/models/networks/grl.py"] = GRL_SRC.format("models/networks/grl.py")
jobs.update({f"{CACHE}/dn_grl_base_c1s{s}.ckpt": GRL_REL.format(f"dn_grl_base_c1s{s}.ckpt") for s in SIGMAS})
jobs.update({f"{CACHE}/{ds}_{n}": RAW.format(ds, n) for ds, ns in IMAGES.items() for n in ns})
with ThreadPoolExecutor(8) as ex:
    list(ex.map(lambda kv: download(kv[1], kv[0]), jobs.items()))
for p in (f"{CACHE}/grl/models/__init__.py", f"{CACHE}/grl/models/networks/__init__.py"):
    open(p, "w").close()

# Xformer gray weights: file ids listed from Drive folder 1Ui_X1JoC5jxgE6k_hK7AZ2qrLo2I_OsD (embeddedfolderview).
XF_FOLDER = "1Ui_X1JoC5jxgE6k_hK7AZ2qrLo2I_OsD"
XF_IDS = {15: "19WHRAbd6m2kTj3larzk-qKYS459rgnrr", 25: "1LkjG1iu39e0FYmIQk3TVtTNfdb4TeN1m",
          50: "1yrpA-VO1XGo8jK53kYphDyadpP0Vf-wk"}


def xf_path(s):
    return f"{CACHE}/xformer/xformer_gray_dn_sigma{s}.pth"


def is_weight_file(path):
    if not os.path.exists(path) or os.path.getsize(path) < 1_000_000:
        return False
    with open(path, "rb") as f:
        return not f.read(16).lstrip().startswith(b"<")  # Drive quota / consent pages are HTML


def fetch_xformer_weights():
    try:
        import gdown
    except Exception as e:
        print(f"gdown unavailable: {e}", flush=True)
        gdown = None
    for s, fid in XF_IDS.items():
        if is_weight_file(xf_path(s)):
            continue
        if gdown is not None:
            try:
                gdown.download(id=fid, output=xf_path(s), quiet=True)
            except Exception as e:
                print(f"gdown file σ {s}: {e}", flush=True)
        if not is_weight_file(xf_path(s)):
            if os.path.exists(xf_path(s)):
                os.remove(xf_path(s))
            download(f"https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t", xf_path(s))
    if gdown is not None and not all(is_weight_file(xf_path(s)) for s in SIGMAS):
        try:  # UNVERIFIED: that download_folder writes the files flat into `output` (gdown layout varies by version)
            gdown.download_folder(id=XF_FOLDER, output=f"{CACHE}/xformer", quiet=True)
        except Exception as e:
            print(f"gdown folder: {e}", flush=True)
    return {s: is_weight_file(xf_path(s)) for s in SIGMAS}


XF_OK = fetch_xformer_weights()
TEST = {ds: [(n, np.asarray(Image.open(f"{CACHE}/{ds}_{n}").convert("L"))) for n in ns] for ds, ns in IMAGES.items()}
print({ds: len(v) for ds, v in TEST.items()}, "Xformer weights:", XF_OK,
      "GRL ckpts:", {s: os.path.getsize(f"{CACHE}/dn_grl_base_c1s{s}.ckpt") if os.path.exists(f"{CACHE}/dn_grl_base_c1s{s}.ckpt") else 0 for s in SIGMAS},
      f"setup {(time.time() - T_START) / 60:.1f} min", flush=True)

In [ ]:
def noisy_of(clean_u8, sigma, seed):
    img = np.float32(clean_u8[..., None]) / 255.
    np.random.seed(seed=seed)
    img += np.random.normal(0, sigma / 255., img.shape)
    return torch.from_numpy(img[..., 0].copy())


def psnr(x, clean_u8):
    a = (x.clamp(0, 1) * 255).round().double()
    return 10 * math.log10(255 ** 2 / (a - torch.from_numpy(clean_u8).double()).pow(2).mean().item())


COLS = ["dataset", "image", "sigma", "seed", "method", "psnr", "seconds"]
rows = []


def save():
    pd.DataFrame(rows, columns=COLS).to_csv(f"{OUT}/scores.csv", index=False)


def run_model(method, build, infer):
    """σ-outer loop; one model in memory at a time; CSV rewritten after each (dataset, σ); stops at DEADLINE_S."""
    last = 0.0
    for s in SIGMAS:
        try:
            net = build(s)
        except Exception:
            print(f"{method} σ {s}: build/load failed\n{traceback.format_exc()}", flush=True)
            continue
        for ds, items in TEST.items():
            for name, clean in items:
                if time.time() - T_START + last > DEADLINE_S:
                    print(f"{method}: deadline reached at σ {s} {ds} {name}", flush=True)
                    save()
                    return
                y = noisy_of(clean, s, SEED)
                t0 = time.time()
                x = infer(net, y)
                last = time.time() - t0
                rows.append(dict(dataset=ds, image=name, sigma=s, seed=SEED, method=method, psnr=psnr(x, clean),
                                 seconds=round(last, 3)))
            save()
            done = [r for r in rows if r["method"] == method and r["sigma"] == s and r["dataset"] == ds]
            print(f"{method} σ {s} {ds}: {np.mean([r['psnr'] for r in done]):.3f} dB, "
                  f"{np.mean([r['seconds'] for r in done]):.1f} s/img, total {(time.time() - T_START) / 3600:.2f} h",
                  flush=True)
        del net

## Xformer
Arch and padding from `evaluate/test_gaussian_gray_denoising.py` (factor 8, reflect pad bottom/right, crop back).
The spatial blocks additionally zero-pad each feature map to a multiple of the 16-px window inside the network.

In [ ]:
XF_KW = dict(inp_channels=1, out_channels=1, dim=48, num_blocks=[2, 4, 4], spatial_num_blocks=[2, 4, 4, 6],
             num_refinement_blocks=4, heads=[1, 2, 4, 8], window_size=[16, 16, 16, 16], ffn_expansion_factor=2.66,
             bias=False, dual_pixel_task=False)  # LayerNorm_type defaults to 'BiasFree', as in the option YAMLs


def build_xformer(s):
    if not XF_OK.get(s):
        raise FileNotFoundError(f"Xformer σ {s} weights not downloaded")
    Xformer = load_module("x_former_arch", f"{CACHE}/xformer/x_former_arch.py").Xformer
    net = Xformer(**XF_KW)
    try:
        ck = torch.load(xf_path(s), map_location="cpu", weights_only=True)
    except Exception:
        ck = torch.load(xf_path(s), map_location="cpu", weights_only=False)
    sd = ck["params"] if "params" in ck else ck  # key from the official test script; .pth contents not inspected
    net.load_state_dict({k.removeprefix("module."): v for k, v in sd.items()}, strict=True)
    return net.eval().requires_grad_(False)


@torch.no_grad()
def xformer(net, y):
    h, w = y.shape  # official: pad = ((h+8)//8)*8 - h if h % 8 else 0, i.e. (-h) % 8
    yp = F.pad(y[None, None], (0, (-w) % 8, 0, (-h) % 8), mode="reflect")
    return net(yp)[0, 0, :h, :w]


try:
    if "Xformer" in RUN:
        run_model("Xformer", build_xformer, xformer)
except Exception:
    print(f"Xformer section failed\n{traceback.format_exc()}", flush=True)
save()

## GRL-B
Tiled inference copied from `engines/base.py::forward_tile` (tile 256, overlap 32, uniform averaging). With
stripes [64, 128] the network's own pad size is 128, so 256-px tiles need no padding. The Lightning `.ckpt` is
loaded as in `tools/trainer.py`: `state_dict`, minus the shared table/index/mask buffers (`convert_checkpoint`),
with the engine's `model.` prefix stripped.

In [ ]:
GRL_KW = dict(upscale=1, in_channels=1, embed_dim=180, img_range=1., img_size=256, upsampler="",
              depths=[4, 4, 8, 8, 8, 4, 4], num_heads_window=[3] * 7, num_heads_stripe=[3] * 7,
              window_size=32,  # grl_test.md GRL-Base dn override; training used 16 (grl_p256.yaml). UNVERIFIED: that
              # this test-time setting (rather than 16 / 4) is what produced the published GRL-B gray numbers
              stripe_size=[64, 128], stripe_groups=[None, None], stripe_shift=True, mlp_ratio=2,
              qkv_proj_type="linear", anchor_proj_type="avgpool", anchor_one_stage=True,
              anchor_window_down_factor=2,  # grl_test.md GRL-Base dn override; training used 4
              out_proj_type="linear", conv_type="1conv", init_method="n", fairscale_checkpoint=False,
              offload_to_cpu=False, double_window=False, stripe_square=False, separable_conv_act=True,
              local_connection=True, use_buffer=True, use_efficient_buffer=True, euclidean_dist=False)
GRL_TILE, GRL_OVERLAP = 256, 32
GRL_BUFFER_KEYS = ("relative_coords_table", "relative_position_index", "attn_mask")
GRL_BUFFER_PREFIXES = ("table_", "index_", "mask_")


class _LenientUnpickler(pickle.Unpickler):
    """Lightning ckpts pickle hyper-parameters (omegaconf / lightning classes); stand-ins suffice for the weights."""

    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except Exception:
            return type(name, (dict,), {"__init__": lambda self, *a, **k: None,
                                        "__setstate__": lambda self, state: None})


LENIENT = types.ModuleType("lenient_pickle")
LENIENT.Unpickler = _LenientUnpickler
LENIENT.load = lambda f, **kw: _LenientUnpickler(f, **kw).load()


def load_ckpt(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"  plain torch.load failed ({e}); retrying with lenient unpickler", flush=True)
        return torch.load(path, map_location="cpu", weights_only=False, pickle_module=LENIENT)


def build_grl(s):
    path = f"{CACHE}/dn_grl_base_c1s{s}.ckpt"
    if not os.path.exists(path) or os.path.getsize(path) < 1_000_000:
        raise FileNotFoundError(path)
    if f"{CACHE}/grl" not in sys.path:
        sys.path.insert(0, f"{CACHE}/grl")
    from models.networks.grl import GRL
    net = GRL(**GRL_KW)
    ck = load_ckpt(path)
    # UNVERIFIED: ckpt contents not inspected; layout inferred from tools/trainer.py (Lightning `state_dict`, engine
    # attribute `model.`) and GRL.convert_checkpoint (`model.table_` keys). The loader accepts either layout.
    sd = ck.get("state_dict", ck)
    sd = sd.get("params", sd)
    if any(k.startswith("model.") for k in sd):
        sd = {k[len("model."):]: v for k, v in sd.items() if k.startswith("model.")}
    sd = {k: v for k, v in sd.items() if torch.is_tensor(v) and not k.startswith(GRL_BUFFER_PREFIXES)
          and not any(b in k for b in GRL_BUFFER_KEYS)}
    missing, unexpected = net.load_state_dict(sd, strict=False)
    buffers = set(dict(net.named_buffers()))
    bad = [k for k in missing if k not in buffers]
    if bad or unexpected:
        raise RuntimeError(f"GRL σ {s} state-dict mismatch: missing {bad[:10]}, unexpected {list(unexpected)[:10]}")
    print(f"GRL σ {s}: loaded {len(sd)} tensors, kept {len(missing)} own buffers", flush=True)
    return net.eval().requires_grad_(False)


@torch.no_grad()
def grl(net, y):
    x = y[None, None]
    h, w = y.shape
    t = min(GRL_TILE, h, w)
    stride = t - GRL_OVERLAP
    hs = list(range(0, h - t, stride)) + [h - t]
    ws = list(range(0, w - t, stride)) + [w - t]
    E, W = torch.zeros_like(x), torch.zeros_like(x)
    for i in hs:
        for j in ws:
            E[..., i:i + t, j:j + t] += net(x[..., i:i + t, j:j + t])
            W[..., i:i + t, j:j + t] += 1
    return (E / W)[0, 0]


try:
    if "GRL-B" in RUN:
        run_model("GRL-B", build_grl, grl)
except Exception:
    print(f"GRL-B section failed\n{traceback.format_exc()}", flush=True)
save()

## Summary

In [ ]:
PUBLISHED = {("Set12", "Xformer"): [33.46, 31.16, 28.10], ("BSD68", "Xformer"): [31.98, 29.55, 26.65],
             ("Set12", "GRL-B"): [33.47, 31.12, 28.03], ("BSD68", "GRL-B"): [32.00, 29.54, 26.60]}
SCORES = pd.DataFrame(rows, columns=COLS)
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
if len(SCORES):
    table = SCORES.groupby(["dataset", "method", "sigma"]).agg(n=("psnr", "size"), psnr=("psnr", "mean"),
                                                              s_per_img=("seconds", "mean")).reset_index()
    table["published"] = [PUBLISHED[(d, m)][[15, 25, 50].index(s)] for d, m, s in
                          zip(table.dataset, table.method, table.sigma)]
    table["delta"] = table.psnr - table.published
    table["complete"] = table.n == table.dataset.map({ds: len(v) for ds, v in TEST.items()})
    print(table.round(3).to_string(index=False))
else:
    print("no scores: both model sections failed")
print(f"total {(time.time() - T_START) / 3600:.2f} h")
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))